# Cursul 2. Colectarea datelor prin API
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

În C1 am citit fișiere și am ajuns la valori din liste și dicționare. Acum cerem date direct de la sursă, cu Python: de la **Banca Mondială (BM)** și de la **Eurostat**.

Pe lângă indicatorii despre locuri, aducem azi și **migrația**: câți oameni dintr-o țară trăiesc în alta. E a doua jumătate a ghicitorii din C1.

**Citește, anticipează, rulează, explică.**

| Pas | Ce facem | Timp |
|---|---|---|
| 1-2 | Pregătire; cum arată o cerere | 10 min |
| 3-5 | Banca Mondială: prima cerere, paginarea, datele proiectului | 30 min |
| 6-7 | Eurostat: JSON-stat, apoi toate regiunile | 25 min |
| 8 | Migrația: un tabel cu două țări pe rând | 15 min |
| 9 | Echipele aleg ce date încearcă | 10 min |

## 1. Pregătire
Folosim același mediu ca în C1. `requests` trimite cereri pe internet, `pandas` face tabele. Amândouă sunt în `requirements.txt`, deci sunt deja instalate. Toate fișierele de azi merg în `data/c2`.

In [ ]:
import requests
import pandas as pd
from pathlib import Path

# Folderul pentru fișierele de azi (dacă există deja, nu se întâmplă nimic)
Path("data/c2").mkdir(exist_ok=True)

## 2. Cum arată o cerere
**Notebook-ul trimite o cerere, serverul răspunde cu date.** Cererea e o adresă web:

`https://api.worldbank.org/v2/country/RO/indicator/SP.POP.TOTL?format=json&date=2022:2024`

| Parte | Ce înseamnă |
|---|---|
| `https://api.worldbank.org/v2` | serverul |
| `/country/RO/indicator/SP.POP.TOTL` | resursa: populația României |
| după `?` | parametrii: formatul și anii |

Deschide adresa în browser. **Anticipează:** câte valori o să vezi pentru trei ani?

## 3. Banca Mondială: prima cerere
Punem parametrii într-un dicționar, iar `requests` construiește adresa. Am cerut intenționat doar 2 valori pe pagină.

In [ ]:
url = "https://api.worldbank.org/v2/country/RO/indicator/SP.POP.TOTL"
parametri = {"format": "json", "date": "2022:2024", "per_page": 2, "page": 1}

raspuns = requests.get(url, params=parametri)   # trimitem cererea
print(raspuns.url)           # adresa construită din url și parametri
print(raspuns.status_code)   # 200 = a mers

`200` înseamnă că serverul a răspuns. `404` ar fi o adresă greșită, `429` prea multe cereri. Acum citim conținutul ca JSON, adică liste și dicționare, ca în C1.

In [ ]:
date = raspuns.json()   # textul răspunsului devine liste și dicționare
date

### 3.1. Recunoaștem structura din C1
Răspunsul e o listă cu două elemente: `[0]` spune câte pagini și câte valori există, `[1]` e lista valorilor.

In [ ]:
meta = date[0]     # informații despre pagini
valori = date[1]   # lista valorilor

print(meta)
valori[0]

In [ ]:
# Drumul până la o valoare, ca în C1:
# afișează numele țării, anul și valoarea primului element din valori

`pd.json_normalize` transformă lista de dicționare într-un tabel. Dicționarele din interior devin coloane cu punct în nume, de exemplu `country.id`.

In [ ]:
# Lista de dicționare devine tabel; păstrăm trei coloane
pd.json_normalize(valori)[["country.id", "date", "value"]]

**Exercițiu.** Fă aceeași cerere pentru Spania (`ES`). Ce se schimbă în adresă?

In [ ]:
# Aceeași cerere pentru Spania: schimbă doar codul țării din adresă

## 4. Paginarea
Am cerut trei ani, dar câte 2 valori pe pagină. **Anticipează:** unde e a treia valoare?

In [ ]:
print("Pagina", meta["page"], "din", meta["pages"])
print("Valori în total:", meta["total"])

In [ ]:
# Cere pagina 2: schimbă "page" în parametri, trimite din nou cererea
# și afișează lista valorilor

Pentru proiect cerem toate valorile dintr-o dată, cu pagini mari (`per_page=1000`), și verificăm că a venit o singură pagină. Funcția de mai jos face pașii de sus, ca să nu-i scriem de fiecare dată.

In [ ]:
def cere_bm(indicator, tari):
    # Țările intră în adresă despărțite prin ";", de exemplu RO;IT;ES
    url = "https://api.worldbank.org/v2/country/" + ";".join(tari) + "/indicator/" + indicator
    parametri = {"format": "json", "date": "2022:2024", "per_page": 1000}
    date = requests.get(url, params=parametri).json()
    print(indicator, "· pagini:", date[0]["pages"])   # trebuie să fie 1
    return pd.json_normalize(date[1])

## 5. Contextul național pentru proiect
Țările de mai jos sunt cele care apar în fișierele SCI regionale ale proiectului. Cerem trei indicatori pentru fiecare.

Atenție la coduri: Banca Mondială și SCI scriu Grecia `GR`. Eurostat o scrie `EL`. Ne întoarcem la asta în C4.

In [ ]:
TARI = ["AL", "AT", "BA", "BE", "BG", "CH", "CY", "CZ", "DE", "DK", "EE", "ES", "FI",
        "FR", "GR", "HR", "HU", "IE", "IS", "IT", "LT", "LU", "LV", "ME", "MK", "MT",
        "NL", "NO", "PL", "PT", "RO", "RS", "SE", "SI", "SK", "TR", "XK"]

# codul indicatorului la Banca Mondială -> numele scurt pe care îl folosim noi
INDICATORI_BM = {"SP.POP.TOTL": "populatie",
                 "NY.GDP.PCAP.CD": "pib_locuitor_usd",
                 "IT.NET.USER.ZS": "internet_pct"}

In [ ]:
tabele = []
for cod, nume in INDICATORI_BM.items():
    tabel = cere_bm(cod, TARI)
    tabel["indicator"] = nume    # notăm ce indicator e în tabel
    tabele.append(tabel)

bm = pd.concat(tabele)           # lipim cele trei tabele unul sub altul
bm = bm[["country.id", "date", "indicator", "value"]]
bm.columns = ["tara", "an", "indicator", "valoare"]
bm.to_csv("data/c2/bm.csv", index=False)
bm.head()

## 6. Eurostat: aceeași idee, alt format
Eurostat are date pe regiuni. Începem cu densitatea județului Cluj (`RO113`) în 2024.

In [ ]:
EUROSTAT = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"

parametri = {"geo": "RO113", "time": 2024, "unit": "PER_KM2"}
cluj = requests.get(EUROSTAT + "demo_r_d3dens", params=parametri).json()
cluj["value"]

### 6.1. Cum citim JSON-stat
Răspunsul nu e un tabel, ci un **cub**: dimensiuni (regiune, an, unitate) și valori puse pe poziții.

| Cheie | Ce ne spune |
|---|---|
| `id` | numele dimensiunilor, în ordine |
| `dimension` | codurile fiecărei dimensiuni și poziția lor |
| `value` | valorile, după poziție |

Cerem două județe. **Anticipează:** la ce poziție va fi Cluj?

In [ ]:
parametri["geo"] = ["RO113", "RO111"]   # Cluj și Bihor
doua = requests.get(EUROSTAT + "demo_r_d3dens", params=parametri).json()

print(doua["dimension"]["geo"]["category"]["index"])  # cod -> poziție
print(doua["value"])                                  # poziție -> valoare

API-ul nu păstrează ordinea în care am cerut. Bihor (`RO111`) a ajuns pe poziția 0. De aceea citim pozițiile, nu presupunem.

### 6.2. Din cub în tabel
Funcția face trei pași. O poziție fără valoare rămâne goală (`None`), nu devine zero.

In [ ]:
def jsonstat_in_tabel(date):
    # 1. Pentru fiecare dimensiune: codurile, în ordinea pozițiilor lor
    coduri = []
    for dim in date["id"]:
        pozitii = date["dimension"][dim]["category"]["index"]   # cod -> poziție
        coduri.append(sorted(pozitii, key=pozitii.get))

    # 2. Toate combinațiile de coduri, câte una pe rând
    tabel = pd.MultiIndex.from_product(coduri, names=date["id"]).to_frame(index=False)

    # 3. Lângă fiecare combinație, valoarea de pe poziția ei
    tabel["valoare"] = [date["value"].get(str(i)) for i in range(len(tabel))]
    return tabel

In [ ]:
jsonstat_in_tabel(doua)

## 7. Toate regiunile Europei
În loc de `geo` folosim `geoLevel`: `nuts2` pentru regiuni mari, `nuts3` pentru județe și echivalentele lor. Le păstrăm separat.

In [ ]:
def cere_eurostat(dataset, parametri):
    date = requests.get(EUROSTAT + dataset, params=parametri).json()
    return jsonstat_in_tabel(date)

Trei indicatori, fiecare cu datasetul și filtrele lui. `sex=T` și `age=TOTAL` înseamnă toată populația, nu o grupă.

In [ ]:
# numele nostru -> (datasetul Eurostat, filtrele lui)
REGIONALE = {
    "populatie":    ("demo_r_pjangrp3", {"sex": "T", "age": "TOTAL", "unit": "NR", "time": 2024}),
    "densitate":    ("demo_r_d3dens",   {"unit": "PER_KM2", "time": 2024}),
    "pib_locuitor": ("nama_10r_3gdp",   {"unit": "PPS_EU27_2020_HAB", "time": 2024}),
}

for nume, (dataset, parametri) in REGIONALE.items():
    for nivel in ["nuts2", "nuts3"]:
        parametri["geoLevel"] = nivel
        tabel = cere_eurostat(dataset, parametri)
        tabel.to_csv(f"data/c2/{nume}_{nivel}.csv", index=False)
        print(nume, nivel, "·", len(tabel), "regiuni ·", tabel["valoare"].count(), "cu valoare")

**Interpretează:** de ce sunt mai puține valori decât regiuni? O regiune fără valoare nu are zero locuitori: sursa nu are dată pentru anul cerut.

## 8. Migrația: două țări pe un rând
Datasetul `migr_pop1ctz` spune câți oameni cu o anumită **cetățenie** trăiesc într-o țară, la 1 ianuarie. E un tabel ca SCI: un rând descrie **o relație între două țări**.

Diferența: migrația are **sens**. Românii din Italia nu sunt același lucru cu italienii din România. SCI nu are sens: RO-IT și IT-RO au aceeași valoare.

Folosim aceeași funcție. **Anticipează:** câte rânduri vor fi, dacă Eurostat are aproape 300 de cetățenii și 45 de țări?

In [ ]:
parametri = {"sex": "T", "age": "TOTAL", "unit": "NR", "time": 2024}
migratie = cere_eurostat("migr_pop1ctz", parametri)
migratie.shape

**Ghicitoarea din C1.** Câți cetățeni români trăiesc în Italia?

In [ ]:
# Rândul în care țara de rezidență (geo) este "IT" și cetățenia (citizen) este "RO".
# Două condiții deodată: fiecare între paranteze, legate cu &

Italia are cei mai mulți români din UE, dar în SCI-ul României era abia pe locul 6. Migrația și legătura socială nu sunt același lucru. De ce diferă le aflăm în cursurile următoare.

**Exercițiu.** Câți români trăiesc în Germania (`DE`) și în Spania (`ES`)?

In [ ]:
# Aceeași selecție ca pentru Italia, cu "DE", apoi cu "ES"

In [ ]:
migratie.to_csv("data/c2/migratie.csv", index=False)

## 9. Echipele: ce date încercați
Alegeți o direcție provizorie și verificați că datele se pot lua. Întrebarea de cercetare o scrieți în C3.

| Direcție | Ce încercați | Dataset Eurostat | Filtre |
|---|---|---|---|
| Migrație | alte cetățenii sau alt an | `migr_pop1ctz` | `sex=T, age=TOTAL, unit=NR` |
| Demografie | ponderea persoanelor de 65+ ani | `demo_r_pjanind3` | `indic_de=PC_Y65_MAX, unit=PC` |
| Educație | studii superioare, 25-64 ani (NUTS2) | `edat_lfse_04` | `isced11=ED5-8, age=Y25-64, sex=T, unit=PC` |
| Piața muncii | rata șomajului (NUTS2) | `lfst_r_lfu3rt` | `isced11=TOTAL, age=Y15-74, sex=T, unit=PC` |

Un indicator nou îl găsiți în [Eurostat Data Browser](https://ec.europa.eu/eurostat/databrowser/): codul datasetului e în titlu, iar filtrele se văd în tabel.

**Exercițiul echipei.** Cereți indicatorul ales pentru un singur județ (`geo`), uitați-vă la răspuns, apoi pentru tot nivelul (`geoLevel`) și salvați-l în `data/c2`.

In [ ]:
# 1. Cererea pentru o singură regiune
# 2. Cererea pentru tot nivelul, cu cere_eurostat
# 3. Salvarea în data/c2

**Fișa echipei**

- Direcția: ...
- Indicatorul încercat (dataset și filtre): ...
- Câte regiuni au valoare: ...

## Ce ducem mai departe
În `data/c2` avem acum `bm.csv`, câte un fișier pe indicator și nivel, și `migratie.csv`. Sursa fiecăruia se vede în celula care l-a produs.

**Poți explica fără cod?** Ce e un parametru? Cum afli dacă ai primit toate paginile? De ce o valoare lipsă nu e zero? De ce migrația are sens, iar SCI nu?

### Documentație
- [Banca Mondială: structura cererilor](https://datahelpdesk.worldbank.org/knowledgebase/articles/898581-api-basic-call-structures)
- [Eurostat: primul exemplu de API](https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-getting-started/api)
- [Specificația JSON-stat](https://json-stat.org/format/)